# Day 2 Development: Quiet & Bright Twin Finder
**Team 5 | Swiss Dwellings | 17 November 2026**

This is our explained adaptation of **Tuesday, sections 0, 1 and 2.1-2.7** of the course `02_deployment.ipynb`. It serves our own trained DenseAE-8 model and shows how a person and another program can use it. The shorter `call_api.ipynb` is the submission's standalone API caller.

The model and matching artifacts are published in [our existing Hugging Face repository](https://huggingface.co/iaac-maai/hackathon-team5-swiss-dwellings/tree/1f410d690ad7e6e6d46442e856f9232183542973). The service is live at https://hackathon-team5-swiss-dwellings.onrender.com. The standalone `call_api.ipynb` contains verified online outputs from 17 November 2026. Rerun this walkthrough for fresh outputs from the deployed service.

## 0. Setup
Use the project Python environment as the notebook kernel and run cells in order. Install packages with `python -m pip install -r 02_model_server/requirements.txt` from the repository root. If installing from this notebook, locate the repository with the next cell first, then uncomment the installation cell and restart the kernel afterwards.

The server uses Gradio, ONNX Runtime and NumPy on CPU. It needs no PyTorch, scikit-learn, training session, or original simulations CSV. The reference catalog is needed for twin search.

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
import time
from pathlib import Path

sys.dont_write_bytecode = True  # Avoid creating notebook import cache files.

import requests
from gradio_client import Client

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "02_model_server" / "app.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or 02_model_server folder.")
SERVER_DIR = ROOT / "02_model_server"
ARTIFACT_DIR = ROOT / "01_autoencoder" / "model_app"
MODEL_URL = "https://hackathon-team5-swiss-dwellings.onrender.com"
print("Server:", MODEL_URL)
print("Artifacts:", ARTIFACT_DIR)
HF_REVISION = "1f410d690ad7e6e6d46442e856f9232183542973"
EXPECTED_MODEL_SHA256 = "884fb2b59c052b72862f0b07e9ef1b3c983c2d9e60494ae4738d35d2c24e77e3"

In [ ]:
# Optional installation after running the setup cell above:
# %pip install -r "$SERVER_DIR/requirements.txt"

## 1. Architecture: what connects to what?

```text
Human using the Gradio page ----> app.py on local Python / Render
Python or HTTP API client ----->          |
                                         +-- order and scale 24 raw features
                                         +-- ONNX: reconstruct + encode
                                         +-- compare error with saved threshold
                                         +-- filter and rank reference apartments
                                         +-- return JSON score, explanation and twins

Hugging Face, pinned release ---> small ONNX download at startup if absent locally
Day 1 model_app/ ---------------> config, examples and derived reference catalog
```

The model is loaded once at startup. A request contains apartment measurements, not the original dataset. Gradio provides the page for people and the HTTP API for programs. Cloudflare, password protection and database storage belong to the next day's work.

The selected model has architecture `24 -> 64 -> 32 -> 8 -> 32 -> 64 -> 24`, 7,904 parameters, threshold **0.179895** and recorded noise-rule ROC AUC **0.7418**. Its full reference catalog contains **44,195 apartments**. Day 2 serves the trained network; it does not retrain or tune it.

## 2.1 Files and responsibilities

| File | Job |
|---|---|
| `app.py` | Load artifacts, validate/scale inputs, run inference, search twins, define Gradio page and API |
| `02_development.ipynb` | This walkthrough, direct checks, local startup, API calls and deployment instructions |
| `call_api.ipynb` | Independent caller with an embedded apartment example and saved results |
| `requirements.txt` | Pinned server and caller dependencies |
| `render.yaml` | Python version, build command, start command, port-ready service configuration |
| `readme.md` | Model explanation, input/output, copyable example and deployment status |
| `../01_autoencoder/model_app/` | Existing ONNX, config, examples, figures and reference catalogs |

Python is pinned in `render.yaml`, so another `.python-version` file is unnecessary. Artifacts stay in the existing Day 1 folder, with no duplicate `models/` folder.

In [ ]:
config = json.loads((ARTIFACT_DIR / "anomaly_ae_config.json").read_text(encoding="utf-8"))
print("Architecture:", config["architecture"])
print("Features:", config["n_features"], "| Latent dimensions:", config["latent_dim"])
print("Threshold:", config["threshold"], "| Recorded test ROC AUC:", config["test_roc_auc"])
for path in sorted(SERVER_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:22s} {path.stat().st_size / 1024:8.1f} KB")

## 2.2 Read the actual server code
**Loading and preprocessing.** The ONNX graph contains the architecture and weights. We order the measurements with `feature_names`, subtract the saved training means, divide by the saved training scales, and cast to float32. We do not fit a scaler on a request.

A local ONNX is checked by SHA-256. If missing, startup downloads it from the **pinned, verified Hugging Face commit**, not a changing `main` branch. The config and catalog are read from the existing Day 1 folder. A different model hash stops startup before serving predictions.

In [ ]:
def show_section(path, start_text=None, end_text=None):
    # Print the real file with line numbers, so explanations track implementation.
    text = Path(path).read_text(encoding="utf-8")
    start = text.index(start_text) if start_text else 0
    end = text.index(end_text, start) if end_text else len(text)
    first_line = text[:start].count("\n") + 1
    for number, line in enumerate(text[start:end].splitlines(), start=first_line):
        print(f"{number:3d} {line}")

show_section(SERVER_DIR / "app.py", end_text="def find_twins(")

**Twin search.** Build a target with quieter acoustic inputs and, if requested, higher sunlight/sky/greenery values. Encode that target, filter real catalog apartments by the requested improvements and approximate size/count, then rank eligible apartments by latent distance.

Area stays within +/-25% and area-record count within +/-2. Acoustic improvement checks indoor and window **daytime traffic** maxima, each more than 10 dBA lower. It does not guarantee lower train noise. Sunlight uses equinox noon, and view filters use sky and greenery. The server returns fewer matches when needed and never discards the filters. The Day 1 notebook's fallback can discard its filters; the server deliberately handles that case more strictly.

In [ ]:
show_section(SERVER_DIR / "app.py", "def find_twins(", "def predict(")

**Prediction, page and API.** `predict` accepts JSON text with all 24 raw measurements and returns a dictionary. It rejects missing/extra names, non-numeric values, NaN/infinity and invalid search options before inference. The Gradio button and `/predict` API call the same function. `/health` is a Gradio API reporting model identity and catalog size, not an ordinary GET `/health` route.

Outputs include the score, threshold, verdict, eight latent values, reconstructed measurements, feature errors and matching apartments. A score above threshold is flagged; neither the score nor its threshold ratio is a probability. A low score does not certify quietness. The server listens on `0.0.0.0` and uses Render's `PORT`, or 7860 locally.

In [ ]:
show_section(SERVER_DIR / "app.py", "def predict(")

## 2.3 Prepare an apartment and call predict without HTTP
We use a saved noisy-rule example with 24 measurements in their original units. JSON carries these numbers directly; base64 is unnecessary for our tabular input. Optional boolean flags choose which domains to improve, and `top_k` requests up to five results.

In [ ]:
payload = json.loads((ARTIFACT_DIR / "examples" / "noisy_apartment_1.json").read_text(encoding="utf-8"))
payload.update(top_k=3, improve_acoustics=True, improve_sunlight=True, improve_view=True)
apartment_json = json.dumps(payload)
print("Query apartment:", payload["apartment_id"])
print("Measurement count:", len(payload["features"]))

Import `app.py` as a module. The import loads the model and catalog without starting a server. Score the four saved examples and keep one direct result for comparison with the HTTP calls. Catalog fingerprints are recomputed with the loaded ONNX, avoiding stale saved fingerprints.

In [ ]:
spec = importlib.util.spec_from_file_location("swiss_model_server", SERVER_DIR / "app.py")
model_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_app)
print(json.dumps(model_app.health(), indent=2))
for path in sorted((ARTIFACT_DIR / "examples").glob("*.json")):
    answer = model_app.predict(path.read_text(encoding="utf-8"))
    print(f"{path.name}: score={answer['score']:.6f}, "
          f"anomaly={answer['is_anomaly']}, twins={len(answer['twins'])}")
direct_result = model_app.predict(apartment_json)
assert model_app.MODEL_SHA256 == EXPECTED_MODEL_SHA256

**Invalid input check.** Omitting a measurement must produce an understandable error rather than filling it with an arbitrary zero. This check calls the shared prediction function directly and does not overwrite model artifacts.

In [ ]:
bad_payload = json.loads(apartment_json)
del bad_payload["features"][config["feature_names"][0]]
try:
    model_app.predict(json.dumps(bad_payload))
except ValueError as error:
    print("Expected validation error:", error)
else:
    raise AssertionError("Missing features should have been rejected.")

## 2.4 Start the local server
The next cell launches the server in the background using the notebook's Python. It waits until Gradio answers and checks for startup failures. On Windows it hides the separate process window. It does not launch anything if `MODEL_URL` points to Render.

Alternatively run `python 02_model_server/app.py` from a terminal in the repository root, then open http://127.0.0.1:7860. Choose a saved example and click **Analyse and find twins**. Stop a terminal server with Ctrl+C. Keep it running while using the standalone `call_api.ipynb`.

In [ ]:
server = None
if MODEL_URL == "http://127.0.0.1:7860":
    try:
        requests.get(MODEL_URL + "/config", timeout=2).raise_for_status()
        print("A server is already listening; the next cell checks its model identity.")
    except requests.RequestException:
        server = subprocess.Popen(
            [sys.executable, "-B", str(SERVER_DIR / "app.py")], cwd=ROOT,
            env={**os.environ, "PORT": "7860", "GRADIO_ANALYTICS_ENABLED": "False"},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
        )
        deadline = time.monotonic() + 60
        while time.monotonic() < deadline:
            if server.poll() is not None:
                raise RuntimeError(server.communicate()[0])
            try:
                requests.get(MODEL_URL + "/config", timeout=2).raise_for_status()
                print("Local server ready:", MODEL_URL)
                break
            except requests.RequestException:
                time.sleep(0.5)
        else:
            server.terminate()
            raise RuntimeError("Startup timed out. Run app.py in a terminal to inspect the error.")
else:
    print("Using remote server:", MODEL_URL)

## 2.5 Call from another program with gradio_client
The client packages the request and waits for the answer. First verify the server's model checksum with `/health`; then send the JSON string to `/predict`. We print the URL and timing, score/verdict and matching IDs. The direct and server scores should agree within a small numerical tolerance.

In [ ]:
client = Client(MODEL_URL, verbose=False)
health_result = client.predict(api_name="/health")
assert health_result["model_sha256"] == model_app.MODEL_SHA256, "The server uses a different model."
start = time.perf_counter()
client_result = client.predict(apartment_json, api_name="/predict")
print("Called:", MODEL_URL)
print(f"API answered in {time.perf_counter() - start:.2f} seconds")
# Keep the output readable while retaining the full response in client_result.
print(json.dumps({k: client_result[k] for k in ["score", "threshold", "is_anomaly", "score_vs_threshold", "matching_candidates"]}, indent=2))
print("Matching apartment IDs:", [t["apartment_id"] for t in client_result["twins"]])
assert abs(client_result["score"] - direct_result["score"]) < 1e-5

### Call the same endpoint with ordinary HTTP
Gradio uses two requests: POST `/gradio_api/call/predict` returns an `event_id`; GET `/gradio_api/call/predict/<event_id>` supplies an event stream. Wait for the completed event and decode its JSON. Heartbeats are not results, and error events must be surfaced.

This is the contract a later Cloudflare function can use. Calls below make real requests to `MODEL_URL` and compare their scores with the Gradio client result.

In [ ]:
def call_model(url, apartment_json, timeout=120):
    # First obtain a ticket for the prediction job.
    ticket = requests.post(url + "/gradio_api/call/predict",
                           json={"data": [apartment_json]}, timeout=timeout)
    ticket.raise_for_status()
    event_id = ticket.json()["event_id"]
    # Read complete SSE events; heartbeat messages are not prediction results.
    event = None
    data = []
    with requests.get(f"{url}/gradio_api/call/predict/{event_id}",
                      stream=True, timeout=timeout) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if line.startswith("event:"):
                event = line[6:].strip()
            elif line.startswith("data:"):
                data.append(line[5:].strip())
            elif not line:
                if event == "complete":
                    return json.loads("\n".join(data))[0]
                if event == "error":
                    raise RuntimeError("Model server error: " + " ".join(data))
                event, data = None, []
    raise RuntimeError("The stream ended without a completed prediction.")

http_result = call_model(MODEL_URL, apartment_json)
print(json.dumps({k: http_result[k] for k in ["score", "threshold", "is_anomaly", "matching_candidates"]}, indent=2))
assert abs(http_result["score"] - client_result["score"]) < 1e-5
print("Direct, client and HTTP scores agree.")

### Interpret the result
Feature errors identify reconstruction mismatches, not physical causes. Smaller latent distance indicates a closer learned fingerprint after filtering; it is not a floor-plan identity or a physical distance. If no candidate passes, an empty twin list is a valid result. The query and catalog are historical simulations, not live property listings.

In [ ]:
largest_errors = sorted(http_result["per_feature_squared_error"].items(), key=lambda pair: pair[1], reverse=True)[:5]
print("Five largest feature reconstruction errors:")
for feature, error in largest_errors:
    print(f"  {feature}: {error:.5f}")
print("\nRecommended apartments:")
for twin in http_result["twins"]:
    f = twin["features"]
    print(twin["apartment_id"], "| distance:", round(twin["latent_distance"], 3),
          "| indoor daytime traffic:", round(f["noise_traffic_day_max"], 2),
          "| area:", round(f["apt_total_area"], 2))
print(http_result["search_note"])

## 2.6 Deploy on Render
1. Push the reviewed Day 2 files and tracked Day 1 config, examples and catalog to the team repository. ONNX weights are ignored by Git; the matching model is already published on Hugging Face.
2. In Render choose **New > Blueprint**, select the team repository, and set Blueprint Path to `02_model_server/render.yaml`. Leave Root Directory blank: this app reads both day folders.
3. Follow the build/runtime logs. Build: `pip install -r 02_model_server/requirements.txt`. Start: `python 02_model_server/app.py`. `render.yaml` pins Python 3.12.8 and selects a free CPU service. It sets the Python version directly rather than needing another file.
4. Copy the **actual assigned URL** into `MODEL_URL` above and into the Day 2 README. Run `call_api.ipynb` against it and save its printed URL and results. A guessed service address or a local call does not demonstrate deployment.

The course suggests a separate server repository; our Blueprint uses the existing team repository while keeping Tuesday's deliverables in `02_model_server/`. The published model is pinned to commit `1f410d690ad7e6e6d46442e856f9232183542973`. Render needs internet access during startup only when the local ONNX is absent. No CSV download or training occurs.

In [ ]:
for filename in ["render.yaml", "requirements.txt"]:
    print(f"\n{filename}")
    print((SERVER_DIR / filename).read_text(encoding="utf-8"))

## 2.7 Wake a sleeping service
The free service sleeps after 15 minutes of inactivity; its next request can take about one minute. Run this helper before the online caller or demo. It retries a limited number of times and reports errors; startup/model failures still require checking Render logs.

In [ ]:
def wake_up(url, max_wait=180):
    """Wait for Gradio, showing the actual URL and reason for each failure."""
    from urllib.parse import urlsplit

    url = url.strip().rstrip("/")
    parts = urlsplit(url)
    if parts.scheme not in ("http", "https") or not parts.netloc or "<" in url:
        raise ValueError("Set MODEL_URL to the actual service URL copied from Render.")
    is_local = parts.hostname in ("127.0.0.1", "localhost", "::1")
    deadline = time.monotonic() + max_wait
    attempt = 0
    last_detail = "No response received."
    print("Checking Gradio:", url + "/config")
    while time.monotonic() < deadline:
        attempt += 1
        remaining = deadline - time.monotonic()
        try:
            response = requests.get(
                url + "/config",
                timeout=(min(10, remaining), min(30, remaining)),
            )
            # These usually indicate a wrong URL/path or access configuration.
            if response.status_code in (401, 403, 404):
                raise RuntimeError(
                    f"HTTP {response.status_code} at {response.url}. "
                    "Use the public Gradio service root URL, without /predict or /config. "
                    "Check that Render is running app.py as a Web Service."
                )
            response.raise_for_status()
            config = response.json()
            if not isinstance(config, dict) or not {"components", "version"}.issubset(config):
                raise ValueError("The response is not a Gradio configuration.")
            print(f"Ready: Gradio {config['version']} at {url}")
            return True
        except (requests.RequestException, ValueError) as error:
            last_detail = f"{type(error).__name__}: {error}"
            print(f"Attempt {attempt}: {last_detail}")
            if is_local and isinstance(error, requests.ConnectionError):
                raise RuntimeError(
                    f"No local server is reachable at {url}. "
                    "For Render, set MODEL_URL to your https://...onrender.com address "
                    "and rerun the setup cell. For local use, start app.py first."
                ) from error
            remaining = deadline - time.monotonic()
            if remaining > 0:
                time.sleep(min(10, remaining))
    raise RuntimeError(
        f"Gradio did not become ready within {max_wait}s at {url}. "
        f"Last failure: {last_detail}. "
        "Open the Render service Logs and check the first startup error."
    )

# Run manually before the online demo:
# wake_up(MODEL_URL)

## Stop the local process created by this notebook
Run this after finishing the local API checks. It closes only the process this notebook created, not an external terminal process or Render service. To use `call_api.ipynb` later, leave a terminal server running or use the deployed URL.

In [ ]:
if server is not None:
    server.terminate()
    server.wait(timeout=15)
    server.stdout.close()
    print("Notebook's local server stopped.")

## Conclusions and submission status
Day 2 demonstrates our trained model behind a Gradio page and API, using the same feature order and scaler as Day 1. Compare direct, Gradio-client and HTTP scores in the outputs above. The standalone caller notebook is the compact evidence file required by the submission guidelines.

**Deployment verified:** the README and both notebooks use https://hackathon-team5-swiss-dwellings.onrender.com; `call_api.ipynb` contains successful online health, Gradio client and HTTP calls. This walkthrough can be rerun against that same URL.

**Limits:** labels are percentile-rule proxies; evaluation anomalies may share buildings with training normals and label cutoffs use the full dataset. ROC AUC 0.7418 and recall 0.2144 therefore describe a preliminary proxy-label evaluation. Area-record count is not official habitable-room count, and measured comfort or exact floor plans are not guaranteed by our features.

**AI use:** Codex adapted the course Tuesday workflow, wrote explanations and comments, and ran local checks. The team reviews model interpretation, deployment settings and real API outputs. The original training dataset and a GPU are unnecessary for serving this model.